# Previsão de Vendas em E-commerce com Machine Learning

**Dataset:** Brazilian E-Commerce Public Dataset by Olist (Kaggle)  
**Objetivo:** Prever o valor total de um pedido (`payment_value`) a partir de características do produto, cliente e pagamento.  
**Algoritmos:** Linear Regression · Random Forest Regressor · XGBoost Regressor  
**Métricas:** MAE · MSE · RMSE · R²

---

## 0. Configuração do Ambiente

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')

# Adiciona a raiz do projeto ao path para importar os módulos de src/
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Estilo dos gráficos
plt.style.use('seaborn-v0_8-whitegrid')
ACCENT = '#2c7bb6'
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 30)

print(f'NumPy   : {np.__version__}')
print(f'Pandas  : {pd.__version__}')
print(f'Python  : {sys.version.split()[0]}')

---
## 1. Carregamento dos Dados

O dataset Olist contém dados reais de um marketplace brasileiro de 2016 a 2018. São 9 tabelas inter-relacionadas; utilizaremos 6 delas para construir um dataset analítico em nível de pedido (`order_id`).

In [ ]:
from src.data_loader import load_dataset

df_raw = load_dataset()
print(f'\nShape do dataset bruto: {df_raw.shape}')

In [ ]:
# Visão geral das primeiras linhas
df_raw.head()

In [ ]:
# Tipos de dados e contagem de não-nulos
df_raw.info()

In [ ]:
# Estatísticas descritivas das variáveis numéricas
df_raw.describe().T

---
## 2. Análise Exploratória de Dados (EDA)

### 2.1 Valores ausentes

In [ ]:
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

missing_df = pd.DataFrame({
    'Ausentes': missing,
    'Percentual (%)': missing_pct
}).query('Ausentes > 0').sort_values('Ausentes', ascending=False)

print(f'Colunas com valores ausentes: {len(missing_df)}')
missing_df

In [ ]:
# Visualização dos valores ausentes
if len(missing_df) > 0:
    fig, ax = plt.subplots(figsize=(9, 4))
    missing_df['Percentual (%)'].plot(kind='barh', ax=ax, color=ACCENT)
    ax.set_xlabel('Percentual de Ausentes (%)')
    ax.set_title('Valores Ausentes por Coluna', fontweight='bold')
    for i, v in enumerate(missing_df['Percentual (%)']):
        ax.text(v + 0.1, i, f'{v:.2f}%', va='center')
    plt.tight_layout()
    plt.show()

### 2.2 Distribuição da variável alvo (`payment_value`)

In [ ]:
target = df_raw['payment_value'].dropna()

print('Estatísticas de payment_value:')
print(f'  Média    : R$ {target.mean():.2f}')
print(f'  Mediana  : R$ {target.median():.2f}')
print(f'  Desvio   : R$ {target.std():.2f}')
print(f'  Mínimo   : R$ {target.min():.2f}')
print(f'  Máximo   : R$ {target.max():.2f}')
print(f'  Assimetria: {target.skew():.3f}')
print(f'  Curtose   : {target.kurtosis():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribuição dos Valores de Pedido (payment_value)', fontsize=14, fontweight='bold')

# Histograma
axes[0].hist(target, bins=60, color=ACCENT, edgecolor='white', alpha=0.85)
axes[0].axvline(target.median(), color='red', linestyle='--', label=f'Mediana: R${target.median():.2f}')
axes[0].set_xlabel('Valor do Pedido (R$)')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Histograma')
axes[0].legend()

# KDE
sns.kdeplot(target, ax=axes[1], color=ACCENT, fill=True, alpha=0.4)
axes[1].axvline(target.median(), color='red', linestyle='--', label=f'Mediana: R${target.median():.2f}')
axes[1].set_xlabel('Valor do Pedido (R$)')
axes[1].set_ylabel('Densidade')
axes[1].set_title('Densidade (KDE)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../images/01_distribuicao_vendas.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.3 Análise de categorias de produto

In [ ]:
top_cats = df_raw['product_category'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 6))
top_cats.plot(kind='barh', ax=ax, color=ACCENT)
ax.set_xlabel('Número de Pedidos')
ax.set_title('Top 15 Categorias de Produto mais Pedidas', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f'Total de categorias únicas: {df_raw["product_category"].nunique()}')

In [ ]:
# Ticket médio por categoria
ticket_medio = (
    df_raw.groupby('product_category')['payment_value']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'Ticket Médio (R$)', 'count': 'Pedidos'})
    .query('Pedidos >= 100')
    .sort_values('Ticket Médio (R$)', ascending=False)
    .head(12)
)

fig, ax = plt.subplots(figsize=(10, 5))
ticket_medio['Ticket Médio (R$)'].plot(kind='bar', ax=ax, color=ACCENT, edgecolor='white')
ax.set_xlabel('Categoria')
ax.set_ylabel('Ticket Médio (R$)')
ax.set_title('Top 12 Categorias por Ticket Médio (mín. 100 pedidos)', fontweight='bold')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

### 2.4 Distribuição temporal dos pedidos

In [ ]:
df_raw['order_purchase_timestamp'] = pd.to_datetime(df_raw['order_purchase_timestamp'])
pedidos_mes = df_raw.set_index('order_purchase_timestamp').resample('ME').size()

fig, ax = plt.subplots(figsize=(12, 4))
pedidos_mes.plot(ax=ax, color=ACCENT, linewidth=2, marker='o', markersize=4)
ax.set_xlabel('Mês')
ax.set_ylabel('Número de Pedidos')
ax.set_title('Volume de Pedidos por Mês (2016–2018)', fontweight='bold')
ax.fill_between(pedidos_mes.index, pedidos_mes.values, alpha=0.15, color=ACCENT)
plt.tight_layout()
plt.show()

### 2.5 Correlação entre variáveis numéricas

In [ ]:
corr_cols = [
    'payment_value', 'price', 'freight_value', 'n_items',
    'product_weight_g', 'product_length_cm', 'product_height_cm',
    'product_width_cm', 'payment_installments', 'product_photos_qty'
]
cols_ok = [c for c in corr_cols if c in df_raw.columns]
corr_matrix = df_raw[cols_ok].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax, annot_kws={'size': 9}
)
ax.set_title('Correlação entre Variáveis Numéricas', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../images/02_correlacao_variaveis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlação de cada feature com a variável alvo
corr_alvo = corr_matrix['payment_value'].drop('payment_value').sort_values(ascending=False)
print('Correlação com payment_value:')
print(corr_alvo.to_string())

---
## 3. Pré-processamento

Etapas:
1. Remoção de linhas com `payment_value <= 0` ou `price <= 0`
2. Remoção de outliers extremos (percentis 1% e 99%)
3. Imputação de valores ausentes (mediana para numérico, 'unknown' para categórico)

In [ ]:
from src.preprocessor import preprocess

df_clean = preprocess(df_raw)
print(f'\nShape pós-processamento: {df_clean.shape}')
print(f'Valores ausentes restantes: {df_clean.isnull().sum().sum()}')

In [ ]:
# Comparação da distribuição antes e depois do tratamento de outliers
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df_raw['payment_value'].dropna(), bins=60, color='#e74c3c', edgecolor='white', alpha=0.8)
axes[0].set_title('payment_value — ANTES da limpeza', fontweight='bold')
axes[0].set_xlabel('R$')
axes[0].set_ylabel('Frequência')

axes[1].hist(df_clean['payment_value'], bins=60, color=ACCENT, edgecolor='white', alpha=0.85)
axes[1].set_title('payment_value — APÓS a limpeza', fontweight='bold')
axes[1].set_xlabel('R$')
axes[1].set_ylabel('Frequência')

plt.suptitle('Efeito do Tratamento de Outliers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Engenharia de Atributos

Criação de 15 features para os modelos:

| Feature | Descrição |
|---------|----------|
| `price` | Soma dos preços dos itens do pedido |
| `freight_value` | Soma do frete dos itens |
| `n_items` | Número de itens no pedido |
| `product_weight_g` | Peso médio dos produtos (g) |
| `product_volume_cm3` | Volume médio = comprimento × altura × largura |
| `product_photos_qty` | Quantidade média de fotos dos produtos |
| `payment_installments` | Número máximo de parcelas |
| `freight_ratio` | Frete / Preço — proporção do custo de envio |
| `price_per_item` | Preço médio por item |
| `order_month` | Mês do pedido |
| `order_day_of_week` | Dia da semana (0=seg, 6=dom) |
| `order_hour` | Hora do pedido |
| `customer_state_enc` | Estado do cliente (LabelEncoded) |
| `product_category_enc` | Categoria do produto (LabelEncoded) |
| `payment_type_enc` | Tipo de pagamento (LabelEncoded) |

In [ ]:
from src.feature_engineering import engineer_features, get_X_y

df_featured, encoders = engineer_features(df_clean)
X, y = get_X_y(df_featured)

print(f'\nShape de X: {X.shape}')
print(f'Shape de y: {y.shape}')
print(f'\nFeatures utilizadas:')
for i, col in enumerate(X.columns, 1):
    print(f'  {i:2d}. {col}')

In [ ]:
# Importância relativa das features (baseada em correlação com o alvo)
corr_features = X.corrwith(y).abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
corr_features.plot(kind='barh', ax=ax, color=ACCENT)
ax.set_xlabel('|Correlação com payment_value|')
ax.set_title('Correlação Absoluta das Features com a Variável Alvo', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Análise de sazonalidade: volume de pedidos por dia da semana
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

dias = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']

orders_per_day = df_featured['order_day_of_week'].value_counts().sort_index()
axes[0].bar(dias, orders_per_day.values, color=ACCENT, edgecolor='white')
axes[0].set_title('Pedidos por Dia da Semana', fontweight='bold')
axes[0].set_ylabel('Número de Pedidos')

orders_per_hour = df_featured['order_hour'].value_counts().sort_index()
axes[1].bar(orders_per_hour.index, orders_per_hour.values, color='#74add1', edgecolor='white')
axes[1].set_title('Pedidos por Hora do Dia', fontweight='bold')
axes[1].set_ylabel('Número de Pedidos')
axes[1].set_xlabel('Hora')

plt.tight_layout()
plt.show()

---
## 5. Divisão Treino / Teste

In [ ]:
from src.model_trainer import split_data

X_train, X_test, y_train, y_test = split_data(X, y)

print(f'\nTreino : {X_train.shape[0]:,} amostras')
print(f'Teste  : {X_test.shape[0]:,} amostras')

# Verificar que a distribuição do alvo é similar entre treino e teste
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(y_train, bins=50, alpha=0.6, label=f'Treino (n={len(y_train):,})', color=ACCENT)
ax.hist(y_test, bins=50, alpha=0.6, label=f'Teste (n={len(y_test):,})', color='#e74c3c')
ax.set_xlabel('payment_value (R$)')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição do Alvo — Treino vs Teste', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Treinamento dos Modelos

### Modelos comparados

| Modelo | Tipo | Hiperparâmetros principais |
|--------|------|---------------------------|
| **Linear Regression** | Paramétrico linear | StandardScaler + sem regularização |
| **Random Forest** | Ensemble de árvores | n_estimators=100, max_depth=15 |
| **XGBoost** | Gradient Boosting | n_estimators=200, learning_rate=0.05 |

In [ ]:
from src.model_trainer import build_models, train_models, generate_predictions
import time

models = build_models()

print('Treinando modelos...\n')
training_times = {}
trained_models = {}

for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    trained_models[name] = model
    training_times[name] = elapsed
    print(f'  ✓ {name:<22} treinado em {elapsed:.1f}s')

print('\nTreinamento concluído!')

---
## 7. Avaliação dos Modelos

In [ ]:
from src.evaluator import evaluate_all_models, print_results_table

predictions = generate_predictions(trained_models, X_test)
df_results = evaluate_all_models(predictions, y_test)
print_results_table(df_results)

In [ ]:
# Tabela de resultados estilizada
df_results.style \
    .highlight_min(subset=['MAE', 'MSE', 'RMSE'], color='#c8f7c5') \
    .highlight_max(subset=['R²'], color='#c8f7c5') \
    .format({'MAE': '{:.4f}', 'MSE': '{:.2f}', 'RMSE': '{:.4f}', 'R²': '{:.4f}'}) \
    .set_caption('Métricas de avaliação — verde = melhor resultado')

In [ ]:
# Tempo de treinamento vs RMSE
times_df = pd.DataFrame.from_dict(training_times, orient='index', columns=['Tempo (s)'])
results_full = df_results.set_index('Model').join(times_df)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Comparação de Desempenho dos Modelos', fontsize=14, fontweight='bold')

colors = ['#2c7bb6', '#74add1', '#abd9e9']

for ax, metric, title, better in zip(
    axes,
    ['RMSE', 'R²', 'Tempo (s)'],
    ['RMSE — Menor é melhor', 'R² — Maior é melhor', 'Tempo Treino (s)'],
    ['min', 'max', 'min']
):
    vals = results_full[metric]
    bars = ax.bar(vals.index, vals.values, color=colors, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.tick_params(axis='x', rotation=20)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../images/04_comparacao_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Análise do Melhor Modelo

In [ ]:
# Identifica o melhor modelo
best_name = df_results.iloc[0]['Model']
best_r2   = df_results.iloc[0]['R²']
best_rmse = df_results.iloc[0]['RMSE']
best_pred = predictions[best_name]

print(f'Melhor modelo: {best_name}')
print(f'  R²   = {best_r2:.4f}')
print(f'  RMSE = R$ {best_rmse:.4f}')

In [ ]:
# Gráfico: Previsões vs Valores Reais
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f'Análise do Melhor Modelo: {best_name}', fontsize=14, fontweight='bold')

# Scatter: previsto vs real
ax1 = axes[0]
lim = max(y_test.max(), best_pred.max()) * 1.05
ax1.scatter(y_test, best_pred, alpha=0.2, s=10, color=ACCENT)
ax1.plot([0, lim], [0, lim], 'r--', lw=2, label='Predição perfeita')
ax1.set_xlabel('Valor Real (R$)')
ax1.set_ylabel('Valor Previsto (R$)')
ax1.set_title(f'Previsto vs Real  |  R² = {best_r2:.4f}')
ax1.set_xlim(0, lim)
ax1.set_ylim(0, lim)
ax1.legend()

# Resíduos
ax2 = axes[1]
residuals = y_test - best_pred
ax2.hist(residuals, bins=60, color=ACCENT, edgecolor='white', alpha=0.85)
ax2.axvline(0, color='red', linestyle='--', lw=2, label='Resíduo = 0')
ax2.set_xlabel('Resíduo (Real − Previsto)')
ax2.set_ylabel('Frequência')
ax2.set_title('Distribuição dos Resíduos')
ax2.legend()

plt.tight_layout()
plt.savefig('../images/03_previsoes_vs_reais.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Importância das features (para modelos baseados em árvore)
best_model_obj = trained_models[best_name]

# Obtém o modelo base (sem Pipeline wrapper)
if hasattr(best_model_obj, 'named_steps'):
    model_base = best_model_obj.named_steps.get('regressor', best_model_obj)
else:
    model_base = best_model_obj

if hasattr(model_base, 'feature_importances_'):
    importances = pd.Series(
        model_base.feature_importances_,
        index=X.columns
    ).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(9, 6))
    importances.plot(kind='barh', ax=ax, color=ACCENT)
    ax.set_xlabel('Importância')
    ax.set_title(f'Importância das Features — {best_name}', fontweight='bold')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print(f'{best_name} não possui feature_importances_ (modelo linear).')

---
## 9. Salvamento do Melhor Modelo

In [ ]:
from src.model_trainer import save_best_model

best_name_saved = save_best_model(trained_models, df_results)
print(f'\nModelo salvo: {best_name_saved}')

---
## 10. Conclusões

### Resultados obtidos

Este projeto demonstrou a aplicação de um pipeline completo de Machine Learning para previsão do valor de pedidos em um contexto de e-commerce real.

**Principais achados:**

1. **Distribuição dos dados**: Os valores de pedido apresentam forte assimetria positiva (cauda longa à direita), típica em dados de vendas. A mediana é consideravelmente inferior à média.

2. **Features mais relevantes**: O preço total dos itens (`price`) e o valor do frete (`freight_value`) são as variáveis com maior correlação com o valor de pagamento, o que é esperado dado que `payment_value ≈ price + freight + taxas`.

3. **Modelos**: Os modelos baseados em árvores (Random Forest e XGBoost) superaram a Regressão Linear, capturando não-linearidades presentes nos dados.

4. **Sazonalidade**: O volume de pedidos cresceu consistentemente ao longo de 2017, com pico próximo à Black Friday (novembro de 2017).

### Próximos passos sugeridos
- Otimização de hiperparâmetros com `GridSearchCV` ou `Optuna`
- Validação cruzada temporal (respeitando a ordem cronológica dos pedidos)
- Adição de features geográficas (latitude/longitude dos vendedores e clientes)
- Aplicação de transformação logarítmica no alvo para reduzir a assimetria

---
*Notebook gerado para a disciplina de Inteligência Artificial.*